# AI Resume Optimizer

An intelligent resume optimization tool that tailors your resume for specific job descriptions using AI.

## How It Works
1. Run all cells in order
2. Mount Google Drive when prompted
3. Enter your Gemini API key
4. Upload or paste your resume
5. Paste a job description
6. Get an AI-optimized resume tailored to the job

## Features
- PDF resume upload or paste text
- AI-powered job description analysis
- Smart clarifying questions
- Resume optimization with keyword matching
- Save optimized resumes by category (ML Engineer, Data Scientist, etc.)
- Resume library stored on Google Drive

In [ ]:
# Install dependencies
!pip install -q gradio google-generativeai PyMuPDF

import gradio as gr
import google.generativeai as genai
import fitz  # PyMuPDF
import json
import os
import re
from datetime import datetime
from typing import Dict, List, Optional

print("All dependencies installed and imported successfully!")

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Set up data directory
DRIVE_PATH = "/content/drive/MyDrive/AI_Resume_Optimizer"
os.makedirs(f"{DRIVE_PATH}/resumes", exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/history", exist_ok=True)

print(f"Data directory: {DRIVE_PATH}")
print("Directories created successfully!")

In [ ]:
# Configuration
GEMINI_MODEL = "gemini-1.5-flash"

CATEGORIES = [
    "ML Engineer",
    "Data Scientist",
    "AI Engineer",
    "Computer Vision",
    "Software Engineer",
    "Custom"
]

print(f"Model: {GEMINI_MODEL}")
print(f"Categories: {', '.join(CATEGORIES)}")
print("Configuration loaded!")

In [ ]:
# PDF Parsing and AI Engine

def parse_pdf(file_path: str) -> str:
    """Extract text from an uploaded PDF file."""
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text.strip()


class ResumeAI:
    """Gemini-powered resume optimization engine."""

    def __init__(self, api_key: str):
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(GEMINI_MODEL)

    def analyze_job_description(self, jd_text: str) -> Dict:
        """Analyze a job description and extract structured requirements."""
        prompt = f"""Analyze this job description and extract structured information.
Return ONLY valid JSON with this exact structure:
{{
    "job_title": "the job title",
    "company": "company name if mentioned",
    "must_have_skills": ["skill1", "skill2"],
    "nice_to_have_skills": ["skill1", "skill2"],
    "key_responsibilities": ["resp1", "resp2"],
    "keywords": ["keyword1", "keyword2"],
    "experience_level": "entry/mid/senior",
    "summary": "2-3 sentence summary of what they're looking for"
}}

Job Description:
{jd_text}"""
        response = self.model.generate_content(prompt)
        text = response.text.strip()
        text = re.sub(r'^```json\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
        return json.loads(text)

    def generate_questions(self, resume_text: str, jd_analysis: Dict) -> List[Dict]:
        """Generate clarifying questions based on resume-JD gaps."""
        prompt = f"""You are a career coach. Compare this resume against job requirements and generate 2-3 targeted questions.

Resume:
{resume_text}

Job Requirements:
- Must have: {', '.join(jd_analysis.get('must_have_skills', []))}
- Nice to have: {', '.join(jd_analysis.get('nice_to_have_skills', []))}
- Key responsibilities: {', '.join(jd_analysis.get('key_responsibilities', []))}

Return ONLY valid JSON array:
[
    {{"id": 1, "question": "the question", "why": "brief reason this matters"}},
    {{"id": 2, "question": "the question", "why": "brief reason this matters"}}
]

Focus on skills gaps that could be addressed by rephrasing experience, which experiences to emphasize, and specific achievements to highlight."""
        response = self.model.generate_content(prompt)
        text = response.text.strip()
        text = re.sub(r'^```json\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
        return json.loads(text)

    def optimize_resume(self, resume_text: str, jd_analysis: Dict, answers: List[Dict]) -> str:
        """Generate an optimized resume tailored to the job description."""
        answers_text = "\n".join([f"Q: {a.get('question', '')} A: {a.get('answer', '')}" for a in answers])

        prompt = f"""You are an expert resume writer. Rewrite this resume to be optimized for the target job.

ORIGINAL RESUME:
{resume_text}

TARGET JOB:
Title: {jd_analysis.get('job_title', 'Unknown')}
Key Skills: {', '.join(jd_analysis.get('must_have_skills', []))}
Responsibilities: {', '.join(jd_analysis.get('key_responsibilities', []))}
Keywords: {', '.join(jd_analysis.get('keywords', []))}

ADDITIONAL CONTEXT FROM CANDIDATE:
{answers_text}

RULES:
1. Keep all factual information accurate - do NOT invent experience
2. Reword and reorganize to highlight relevant skills and experience
3. Naturally incorporate keywords from the job description
4. Tailor the professional summary to this specific role
5. Emphasize achievements that align with job requirements
6. Use strong action verbs and quantify achievements
7. Output the complete optimized resume as clean text"""

        response = self.model.generate_content(prompt)
        return response.text


# Global AI engine instance (initialized when API key is set)
ai_engine = None
print("PDF parser and AI engine defined!")

In [ ]:
# Resume Storage Manager

class ResumeStorage:
    """Manages saving and loading optimized resumes on Google Drive."""

    def __init__(self, base_path: str):
        self.base_path = base_path

    def save(self, category: str, optimized_text: str, job_title: str, company: str) -> str:
        """Save an optimized resume under a category folder."""
        cat_dir = os.path.join(self.base_path, "resumes", category.lower().replace(" ", "_"))
        os.makedirs(cat_dir, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        safe_title = job_title.replace(" ", "_")[:30]
        filename = f"{timestamp}_{safe_title}.txt"
        filepath = os.path.join(cat_dir, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(optimized_text)

        # Append to history log
        history_path = os.path.join(self.base_path, "history", "log.json")
        history = []
        if os.path.exists(history_path):
            with open(history_path, "r") as f:
                history = json.load(f)

        history.append({
            "timestamp": datetime.now().isoformat(),
            "category": category,
            "job_title": job_title,
            "company": company,
            "filename": filename
        })

        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

        return filename

    def list_by_category(self) -> Dict:
        """List all saved resumes grouped by category."""
        resumes_dir = os.path.join(self.base_path, "resumes")
        categories = {}
        if not os.path.exists(resumes_dir):
            return categories

        for cat in sorted(os.listdir(resumes_dir)):
            cat_path = os.path.join(resumes_dir, cat)
            if os.path.isdir(cat_path):
                files = []
                for fname in sorted(os.listdir(cat_path), reverse=True):
                    fpath = os.path.join(cat_path, fname)
                    with open(fpath, "r", encoding="utf-8") as fh:
                        files.append({"name": fname, "content": fh.read()})
                categories[cat] = files

        return categories

    def get_history(self) -> List[Dict]:
        """Load optimization history."""
        history_path = os.path.join(self.base_path, "history", "log.json")
        if os.path.exists(history_path):
            with open(history_path, "r") as f:
                return json.load(f)
        return []


storage = ResumeStorage(DRIVE_PATH)
print(f"Storage manager ready. Saving to: {DRIVE_PATH}")

In [ ]:
# Application State

class AppState:
    """Manages current session state."""
    def __init__(self):
        self.resume_text = ""
        self.jd_analysis = None
        self.questions = []

state = AppState()
print("Application state initialized!")

In [ ]:
# Event Handlers

def on_set_api_key(api_key):
    """Initialize AI engine with API key."""
    global ai_engine
    if not api_key or not api_key.strip():
        return "Please enter a valid API key."
    try:
        ai_engine = ResumeAI(api_key.strip())
        return "API key set successfully! You can now optimize resumes."
    except Exception as e:
        return f"Error setting API key: {e}"

def on_upload_pdf(file):
    """Handle PDF file upload."""
    if file is None:
        return ""
    try:
        text = parse_pdf(file.name)
        state.resume_text = text
        return text
    except Exception as e:
        return f"Error parsing PDF: {e}"

def on_analyze_and_question(resume_text, jd_text):
    """Analyze JD and generate clarifying questions."""
    if not ai_engine:
        return "Please set your API key first.", "", gr.update(visible=False), gr.update(), gr.update(), gr.update()
    if not resume_text.strip():
        return "Please enter or upload your resume first.", "", gr.update(visible=False), gr.update(), gr.update(), gr.update()
    if not jd_text.strip():
        return "Please enter a job description.", "", gr.update(visible=False), gr.update(), gr.update(), gr.update()

    state.resume_text = resume_text

    try:
        # Step 1: Analyze JD
        analysis = ai_engine.analyze_job_description(jd_text)
        state.jd_analysis = analysis

        # Format analysis display
        skills = ', '.join(analysis.get('must_have_skills', []))
        nice = ', '.join(analysis.get('nice_to_have_skills', []))
        resps = '\n'.join([f"- {r}" for r in analysis.get('key_responsibilities', [])])

        analysis_md = f"""### Job Analysis Complete!

**Position:** {analysis.get('job_title', 'N/A')}
**Company:** {analysis.get('company', 'N/A')}
**Level:** {analysis.get('experience_level', 'N/A')}

**Must-Have Skills:** {skills}

**Nice-to-Have:** {nice}

**Key Responsibilities:**
{resps}

**Summary:** {analysis.get('summary', '')}
"""

        # Step 2: Generate questions
        questions = ai_engine.generate_questions(resume_text, analysis)
        state.questions = questions

        q1 = questions[0]['question'] if len(questions) > 0 else ""
        q2 = questions[1]['question'] if len(questions) > 1 else ""
        q3 = questions[2]['question'] if len(questions) > 2 else ""

        return (
            analysis_md,
            "Answer the questions below to help tailor your resume:",
            gr.update(visible=True),
            gr.update(label=f"Q1: {q1}" if q1 else gr.update(visible=False)),
            gr.update(label=f"Q2: {q2}" if q2 else gr.update(visible=False)),
            gr.update(label=f"Q3: {q3}" if q3 else gr.update(visible=False))
        )
    except Exception as e:
        return f"Error during analysis: {e}", "", gr.update(visible=False), gr.update(), gr.update(), gr.update()

def on_optimize(resume_text, a1, a2, a3):
    """Generate optimized resume."""
    if not ai_engine:
        return "Please set your API key first."
    if not state.jd_analysis:
        return "Please analyze a job description first."

    state.resume_text = resume_text
    answers = []
    for i, ans in enumerate([a1, a2, a3]):
        if ans and ans.strip() and i < len(state.questions):
            answers.append({
                "question": state.questions[i]["question"],
                "answer": ans.strip()
            })

    try:
        optimized = ai_engine.optimize_resume(state.resume_text, state.jd_analysis, answers)
        return optimized
    except Exception as e:
        return f"Error optimizing resume: {e}"

def on_save(category, optimized_text):
    """Save the optimized resume."""
    if not optimized_text or not optimized_text.strip():
        return "No optimized resume to save. Run optimization first."
    if not category:
        return "Please select a category."

    job_title = state.jd_analysis.get("job_title", "unknown") if state.jd_analysis else "unknown"
    company = state.jd_analysis.get("company", "unknown") if state.jd_analysis else "unknown"

    try:
        filename = storage.save(category, optimized_text, job_title, company)
        return f"Saved successfully as: {filename} (Category: {category})"
    except Exception as e:
        return f"Error saving: {e}"

def on_load_library():
    """Load and display saved resumes."""
    categories = storage.list_by_category()
    if not categories:
        return "No saved resumes yet. Optimize and save a resume first!"

    md = ""
    for cat, files in categories.items():
        cat_display = cat.replace('_', ' ').title()
        md += f"\n## {cat_display} ({len(files)} resumes)\n"
        for f in files[:3]:  # Show latest 3 per category
            md += f"\n### {f['name']}\n"
            preview = f['content'][:600]
            md += f"```\n{preview}\n{'...' if len(f['content']) > 600 else ''}\n```\n"
    return md

print("Event handlers defined!")


In [ ]:
# Gradio UI

def build_app():
    with gr.Blocks(
        title="AI Resume Optimizer",
        theme=gr.themes.Soft(primary_hue="teal", secondary_hue="rose", neutral_hue="slate")
    ) as app:

        gr.Markdown("# AI Resume Optimizer\nTailor your resume for any job description using AI")

        with gr.Tabs():
            # === TAB 1: OPTIMIZE ===
            with gr.Tab("Optimize Resume"):
                with gr.Row():
                    with gr.Column(scale=1):
                        gr.Markdown("### Step 1: Your Resume")
                        pdf_upload = gr.File(label="Upload PDF (optional)", file_types=[".pdf"])
                        resume_input = gr.Textbox(
                            label="Resume Text",
                            placeholder="Upload a PDF above or paste your resume here...",
                            lines=12
                        )
                    with gr.Column(scale=1):
                        gr.Markdown("### Step 2: Job Description")
                        jd_input = gr.Textbox(
                            label="Job Description",
                            placeholder="Paste the complete job description here...",
                            lines=14
                        )

                analyze_btn = gr.Button("Analyze Job Description", variant="primary", size="lg")

                # Analysis results
                analysis_output = gr.Markdown("")
                questions_label = gr.Markdown("")

                # Questions section (hidden until analysis runs)
                questions_section = gr.Group(visible=False)
                with questions_section:
                    answer1 = gr.Textbox(label="Q1", lines=2)
                    answer2 = gr.Textbox(label="Q2", lines=2)
                    answer3 = gr.Textbox(label="Q3", lines=2)
                    optimize_btn = gr.Button("Optimize My Resume", variant="primary", size="lg")

                # Optimized result
                gr.Markdown("### Optimized Resume")
                optimized_output = gr.Textbox(label="Optimized Resume", lines=15, interactive=True)

                # Save section
                with gr.Row():
                    category_dropdown = gr.Dropdown(
                        choices=CATEGORIES,
                        label="Save to Category",
                        scale=2
                    )
                    save_btn = gr.Button("Save Resume", variant="secondary", scale=1)
                save_status = gr.Textbox(label="Save Status", interactive=False)

            # === TAB 2: LIBRARY ===
            with gr.Tab("Resume Library"):
                refresh_btn = gr.Button("Load Saved Resumes", variant="primary")
                library_output = gr.Markdown("Click 'Load Saved Resumes' to view your resume library.")

            # === TAB 3: SETTINGS ===
            with gr.Tab("Settings"):
                gr.Markdown("### API Configuration")
                api_key_input = gr.Textbox(
                    label="Gemini API Key",
                    type="password",
                    placeholder="Enter your Gemini API key..."
                )
                api_key_btn = gr.Button("Set API Key", variant="primary")
                api_status = gr.Textbox(label="Status", interactive=False, value="API key not set")

                gr.Markdown(f"### Data Location\n`{DRIVE_PATH}`\n\nYour resumes are saved to Google Drive automatically.")

        # === EVENT WIRING ===
        api_key_btn.click(fn=on_set_api_key, inputs=[api_key_input], outputs=[api_status])

        pdf_upload.change(fn=on_upload_pdf, inputs=[pdf_upload], outputs=[resume_input])

        analyze_btn.click(
            fn=on_analyze_and_question,
            inputs=[resume_input, jd_input],
            outputs=[analysis_output, questions_label, questions_section, answer1, answer2, answer3]
        )

        optimize_btn.click(
            fn=on_optimize,
            inputs=[resume_input, answer1, answer2, answer3],
            outputs=[optimized_output]
        )

        save_btn.click(
            fn=on_save,
            inputs=[category_dropdown, optimized_output],
            outputs=[save_status]
        )

        refresh_btn.click(fn=on_load_library, outputs=[library_output])

    return app

app = build_app()
print("Gradio app built!")


In [ ]:
# Launch the app
app.launch(share=True, debug=True)
